# FOMC Monetary Policy Stance Q&A System — RAG Pipeline

A lightweight RAG system that enables **Mistral 7B** to produce grounded, accurate answers about Federal Reserve monetary policy by retrieving relevant passages from FOMC minutes and statements (2020–2025).

**Pipeline Architecture:**
1. Data Ingestion & Preprocessing
2. Chunking (paragraph-aware, 4 size configurations)
3. Embedding & Indexing (FAISS)
4. Retrieval
5. Generation (Mistral 7B)
6. Evaluation (with vs without RAG, chunk size comparison)
7. *(Optional)* RoBERTa Stance Enrichment

---


## 0. Setup & Installations

In [ ]:
# Run this cell first — installs all dependencies
!pip install -q datasets sentence-transformers transformers accelerate bitsandbytes torch rouge-score google-generativeai

# FAISS: try GPU first, fall back to CPU
import subprocess, sys
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu"])
    print("Installed faiss-gpu")
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    print("faiss-gpu not available — installed faiss-cpu (perfectly fine for our corpus size)")

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from huggingface_hub import login
login(token="[hf_login_token_goes_here]")

print("Setup complete.")

In [ ]:
# Mount Google Drive and cache HuggingFace models there
# This means models only download once — subsequent sessions load from Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Colab Notebooks/hf_cache', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/Colab Notebooks/hf_cache'

## Phase 1: Data Ingestion & Preprocessing

Load FOMC statements and minutes from the `vtasca/fomc-statements-minutes` HuggingFace dataset, filter to 2020–2025, and clean the text.

**Cleaning steps:**
- Fix encoding artifacts (corrupted em/en dashes, smart quotes, etc.)
- Remove attendance lists and footnotes (boilerplate in minutes)
- Remove implementation note references and media inquiry lines
- Normalize whitespace
- Preserve section headers (useful context for retrieval)


In [ ]:
def fix_encoding(text):
    """
    Fix common encoding artifacts in FOMC documents.
    These occur when UTF-8 text is misinterpreted as Latin-1/Windows-1252.
    """
    if not isinstance(text, str):
        return ""

    # Try to fix mojibake by re-encoding
    try:
        fixed = text.encode('latin-1').decode('utf-8')
        return fixed
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass

    # Manual replacements for common patterns
    replacements = {
        '\u00e2\u0080\u0093': '–',
        '\u00e2\u0080\u0094': '—',
        '\u00e2\u0080\u0099': "'",
        '\u00e2\u0080\u009c': '"',
        '\u00e2\u0080\u009d': '"',
        '\u00e2\u0080\u0098': "'",
        '\u00c2\u00a0': ' ',
        'â\x80\x93': '–',
        'â\x80\x94': '—',
        'â\x80\x99': "'",
        'â\x80\x9c': '"',
        'â\x80\x9d': '"',
        'â': '—',
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text


def clean_text(text):
    """
    Clean raw FOMC document text.
    Fixes encoding, removes boilerplate, preserves section headers.
    """
    if not isinstance(text, str):
        return ""

    text = fix_encoding(text)
    text = re.sub(r'(?<=[a-zA-Z])\d+(?=\s)', '', text)

    attendance_patterns = [
        r'\nAttendance\s*\n.*$',
        r'\nNotation Vote\s*\n.*$',
        r'\n_+\s*\n.*$',
    ]
    for pattern in attendance_patterns:
        text = re.sub(pattern, '', text, flags=re.DOTALL)

    text = re.sub(r'Implementation Note issued.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'For media inquiries.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()


def load_fomc_data(start_year=2020, end_year=2025):
    """
    Load and preprocess FOMC statements and minutes.
    Returns a cleaned DataFrame filtered to the specified year range.
    """
    from datasets import load_dataset

    print("Loading FOMC dataset from HuggingFace...")
    ds = load_dataset("vtasca/fomc-statements-minutes")
    df = ds['train'].to_pandas()

    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
    df['date'] = pd.to_datetime(df['date'])
    df['release_date'] = pd.to_datetime(df['release_date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

    df = df[(df['year'] >= start_year) & (df['year'] <= end_year)].copy()

    print(f"Filtered to {start_year}-{end_year}: {len(df)} documents")
    print(f"  - Statements: {len(df[df['type'] == 'Statement'])}")
    print(f"  - Minutes: {len(df[df['type'] == 'Minute'])}")

    print("Cleaning text...")
    df['text'] = df['text'].apply(clean_text)

    df['doc_id'] = df.apply(
        lambda row: f"{row['date'].strftime('%Y-%m-%d')}_{row['type'].lower()}",
        axis=1
    )

    df = df.reset_index(drop=True)

    print(f"\nData loaded successfully. {len(df)} documents ready.")
    print(f"Date range: {df['date'].min().strftime('%Y-%m-%d')} to "
          f"{df['date'].max().strftime('%Y-%m-%d')}")

    return df

In [ ]:
# Load the data
df = load_fomc_data(start_year=2020, end_year=2025)

### Verify Encoding Fix

In [ ]:
# Quick check for remaining encoding artifacts
artifacts = ['â\x80', 'â\x93', 'â\x94', 'Ã¢', 'Ã©']

print("Checking for remaining encoding artifacts...")
issues_found = False

for _, row in df.iterrows():
    text = row['text']
    for artifact in artifacts:
        if artifact in text:
            print(f"  Found '{artifact}' in {row['doc_id']}")
            issues_found = True

    if re.search(r'(?<![a-zA-Z])â(?![a-zA-Z])', text):
        matches = [(m.start(), text[max(0,m.start()-20):m.end()+20])
                   for m in re.finditer(r'(?<![a-zA-Z])â(?![a-zA-Z])', text)]
        for pos, context in matches[:2]:
            print(f"  Possible artifact in {row['doc_id']}: ...{context}...")
            issues_found = True

if not issues_found:
    print("No encoding artifacts found — text is clean!")
else:
    print("\nSome artifacts remain — may need additional cleaning rules.")

### Explore the Data

In [ ]:
def explore_data(df):
    """Print summary statistics and sample texts."""
    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)

    print(f"\nTotal documents: {len(df)}")
    print(f"Date range: {df['date'].min().strftime('%Y-%m-%d')} to "
          f"{df['date'].max().strftime('%Y-%m-%d')}")

    print("\nDocuments per year:")
    print(df.groupby(['year', 'type']).size().unstack(fill_value=0))

    df_temp = df.copy()
    df_temp['text_length'] = df_temp['text'].str.len()
    df_temp['word_count'] = df_temp['text'].str.split().str.len()

    print("\nText length statistics (characters):")
    for doc_type in df_temp['type'].unique():
        subset = df_temp[df_temp['type'] == doc_type]
        print(f"\n  {doc_type}:")
        print(f"    Mean:   {subset['text_length'].mean():,.0f}")
        print(f"    Median: {subset['text_length'].median():,.0f}")
        print(f"    Min:    {subset['text_length'].min():,.0f}")
        print(f"    Max:    {subset['text_length'].max():,.0f}")

    print("\nWord count statistics:")
    for doc_type in df_temp['type'].unique():
        subset = df_temp[df_temp['type'] == doc_type]
        print(f"\n  {doc_type}:")
        print(f"    Mean:   {subset['word_count'].mean():,.0f}")
        print(f"    Median: {subset['word_count'].median():,.0f}")
        print(f"    Min:    {subset['word_count'].min():,.0f}")
        print(f"    Max:    {subset['word_count'].max():,.0f}")

    print("\n" + "=" * 60)
    print("SAMPLE STATEMENT (first 500 chars):")
    print("=" * 60)
    sample_stmt = df[df['type'] == 'Statement'].iloc[0]
    print(f"Date: {sample_stmt['date'].strftime('%Y-%m-%d')}")
    print(sample_stmt['text'][:500])

    print("\n" + "=" * 60)
    print("SAMPLE MINUTES (first 500 chars):")
    print("=" * 60)
    sample_min = df[df['type'] == 'Minute'].iloc[0]
    print(f"Date: {sample_min['date'].strftime('%Y-%m-%d')}")
    print(sample_min['text'][:500])

explore_data(df)

## Phase 2: Chunking

We use **paragraph-aware chunking** which preserves the natural topical structure of FOMC minutes (each paragraph typically covers one discussion topic).

To study the effect of chunk granularity on retrieval quality, we generate **four chunk size configurations**: 256, 512, 768, and 1024 words. This spans the range from highly specific (256) to broad context (1024), allowing us to identify the optimal trade-off between:
- **Precision**: Smaller chunks embed a tighter topic, matching specific queries better
- **Context**: Larger chunks preserve more surrounding reasoning, improving generation quality

All four configurations will be embedded and indexed separately, then compared during evaluation (Phase 6).


In [ ]:
# --- Chunking helper functions ---

def _get_overlap_text(text, num_words):
    """Extract the last num_words from a text for chunk overlap."""
    words = text.split()
    overlap_words = words[-num_words:] if len(words) > num_words else words
    return ' '.join(overlap_words)


def _split_large_paragraph(text, chunk_size, chunk_overlap):
    """Split an oversized paragraph into chunks by sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = []
    current_word_count = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        if current_word_count + sentence_words > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))

            if chunk_overlap > 0:
                overlap = _get_overlap_text(' '.join(current_chunk), chunk_overlap)
                current_chunk = [overlap, sentence]
                current_word_count = len(overlap.split()) + sentence_words
            else:
                current_chunk = [sentence]
                current_word_count = sentence_words
        else:
            current_chunk.append(sentence)
            current_word_count += sentence_words

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks


def _chunk_minutes_paragraph(text, chunk_size=512, chunk_overlap=50):
    """
    Paragraph-aware chunking for FOMC minutes.
    Respects natural paragraph boundaries where possible.
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    chunks = []
    current_chunk_parts = []
    current_word_count = 0

    for para in paragraphs:
        para_word_count = len(para.split())

        if para_word_count > chunk_size:
            if current_chunk_parts:
                chunks.append('\n\n'.join(current_chunk_parts))
                current_chunk_parts = []
                current_word_count = 0
            sub_chunks = _split_large_paragraph(para, chunk_size, chunk_overlap)
            chunks.extend(sub_chunks)

        elif current_word_count + para_word_count > chunk_size:
            chunks.append('\n\n'.join(current_chunk_parts))

            if chunk_overlap > 0 and current_chunk_parts:
                overlap_text = _get_overlap_text(current_chunk_parts[-1], chunk_overlap)
                current_chunk_parts = [overlap_text, para]
                current_word_count = len(overlap_text.split()) + para_word_count
            else:
                current_chunk_parts = [para]
                current_word_count = para_word_count
        else:
            current_chunk_parts.append(para)
            current_word_count += para_word_count

    if current_chunk_parts:
        chunks.append('\n\n'.join(current_chunk_parts))

    return chunks


def chunk_documents(df, chunk_size=512, chunk_overlap=50):
    """
    Split all FOMC documents into retrievable chunks with metadata.
    Uses paragraph-aware chunking for minutes; keeps statements whole.
    """
    all_chunks = []

    for _, row in df.iterrows():
        doc_type = row['type']
        meeting_date = row['date'].strftime('%Y-%m-%d')
        doc_id = row['doc_id']
        text = row['text']

        if doc_type == 'Statement':
            chunks = [text]
        else:
            chunks = _chunk_minutes_paragraph(text, chunk_size, chunk_overlap)

        for i, chunk_text in enumerate(chunks):
            all_chunks.append({
                'text': chunk_text,
                'doc_id': doc_id,
                'meeting_date': meeting_date,
                'doc_type': doc_type,
                'chunk_index': i,
                'total_chunks': len(chunks),
            })

    chunk_word_counts = [len(c['text'].split()) for c in all_chunks]
    print(f"Chunking (size={chunk_size}, overlap={chunk_overlap}): "
          f"{len(all_chunks)} chunks "
          f"({sum(1 for c in all_chunks if c['doc_type'] == 'Statement')} statements, "
          f"{sum(1 for c in all_chunks if c['doc_type'] == 'Minute')} minutes) | "
          f"words: mean={np.mean(chunk_word_counts):.0f}, "
          f"median={np.median(chunk_word_counts):.0f}, "
          f"min={np.min(chunk_word_counts)}, max={np.max(chunk_word_counts)}")

    return all_chunks


print("Chunking functions defined.")

### Generate All Chunk Size Configurations

In [ ]:
# Define our four chunk size configurations
# Overlap scales proportionally (~10% of chunk size)
CHUNK_CONFIGS = {
    256:  {'chunk_size': 256,  'chunk_overlap': 25},
    512:  {'chunk_size': 512,  'chunk_overlap': 50},
    768:  {'chunk_size': 768,  'chunk_overlap': 75},
    1024: {'chunk_size': 1024, 'chunk_overlap': 100},
}

# Generate chunks for all configurations
all_chunk_sets = {}

print("Generating chunks for all configurations...")
print("-" * 80)

for size, config in CHUNK_CONFIGS.items():
    all_chunk_sets[size] = chunk_documents(df, **config)

print("-" * 80)
print(f"\nGenerated {len(all_chunk_sets)} chunk sets, ready for embedding in Phase 3.")
print(f"Primary configuration: 512 words (will be used as default)")

### Inspect & Compare Chunk Configurations

In [ ]:
def inspect_chunks(chunks, label="", num_samples=2):
    """Show sample chunks with metadata."""
    print(f"\n{'='*60}")
    print(f"CHUNK INSPECTION — {label}")
    print(f"{'='*60}")

    stmt_chunks = [c for c in chunks if c['doc_type'] == 'Statement']
    if stmt_chunks:
        sample = stmt_chunks[0]
        print(f"\n--- STATEMENT (Meeting: {sample['meeting_date']}) ---")
        print(f"Words: {len(sample['text'].split())}")
        print(f"Preview: {sample['text'][:200]}...")

    min_chunks = [c for c in chunks if c['doc_type'] == 'Minute']
    if min_chunks:
        sample_date = min_chunks[0]['meeting_date']
        meeting_chunks = [c for c in min_chunks if c['meeting_date'] == sample_date]

        print(f"\n--- MINUTES (Meeting: {sample_date}, {len(meeting_chunks)} chunks) ---")
        for c in meeting_chunks[:num_samples]:
            print(f"\n  Chunk {c['chunk_index']+1}/{c['total_chunks']} "
                  f"({len(c['text'].split())} words):")
            print(f"  {c['text'][:150]}...")

# Inspect 256 vs 1024 to see the extremes
inspect_chunks(all_chunk_sets[256], "256 words (most specific)")
inspect_chunks(all_chunk_sets[1024], "1024 words (most context)")

### Chunk Size Distribution Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

summary_data = []

for ax, (size, chunks), color in zip(axes.flat, all_chunk_sets.items(), colors):
    # Only plot minutes chunks (statements are always 1 chunk)
    min_wc = [len(c['text'].split()) for c in chunks if c['doc_type'] == 'Minute']

    ax.hist(min_wc, bins=30, edgecolor='black', alpha=0.7, color=color)
    ax.axvline(x=size, color='red', linestyle='--', linewidth=2, label=f'Target ({size})')
    ax.set_xlabel('Words per Chunk')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Chunk Size = {size} words ({len(min_wc)} minutes chunks)')
    ax.legend()

    summary_data.append({
        'Target Size': size,
        'Total Chunks': len(chunks),
        'Minutes Chunks': len(min_wc),
        'Mean Words': np.mean(min_wc),
        'Median Words': np.median(min_wc),
        'Std Dev': np.std(min_wc),
        'Min': np.min(min_wc),
        'Max': np.max(min_wc),
    })

plt.tight_layout()
plt.show()

# Summary table
summary_df = pd.DataFrame(summary_data)
print("\nChunk Configuration Summary (Minutes only):")
print("=" * 90)
print(summary_df.to_string(index=False, float_format='{:.0f}'.format))

In [ ]:
# Set primary chunk set (512) for the main pipeline
# Other sizes stored in all_chunk_sets for evaluation comparison
chunks = all_chunk_sets[512]
print(f"Primary chunk set: {len(chunks)} chunks (512 words, paragraph-aware)")
print(f"All chunk sets available: {list(all_chunk_sets.keys())} words")

---

## Phase 3: Embedding & Indexing (FAISS)

Convert text chunks into vector embeddings and build FAISS indices for fast similarity search.

We compare **two embedding models** to evaluate how model quality affects retrieval:

| Model | Parameters | Dimensions | Strengths |
|---|---|---|---|
| `intfloat/e5-small-v2` | 33M | 384 | Lightweight, fast, 100% top-5 retrieval accuracy in benchmarks |
| `BAAI/bge-base-en-v1.5` | 109M | 768 | Retrieval-optimized with hard negative mining, better at distinguishing similar passages |

**Why these two?** They represent different points on the size-vs-quality trade-off. E5-small is 3x smaller but achieves excellent retrieval recall. BGE-base is heavier but specifically trained to distinguish between deceptively similar documents — important for FOMC minutes where many paragraphs discuss overlapping topics in similar language.

**Important:** Both models require query prefixes for optimal performance:
- E5: Prefix queries with `"query: "` and documents with `"passage: "`
- BGE: Prefix queries with `"Represent this sentence for searching relevant passages: "`

**FAISS Index:** `IndexFlatIP` (exact inner product search with normalized vectors = cosine similarity)

Combined with 4 chunk sizes, this gives us **8 configurations** to compare during evaluation.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import time

# --- Model configuration ---
# Each model has specific prefix requirements for optimal retrieval

MODEL_CONFIGS = {
    'e5-small': {
        'model_name': 'intfloat/e5-small-v2',
        'query_prefix': 'query: ',
        'passage_prefix': 'passage: ',
    },
    'bge-base': {
        'model_name': 'BAAI/bge-base-en-v1.5',
        'query_prefix': 'Represent this sentence for searching relevant passages: ',
        'passage_prefix': '',  # BGE doesn't require document prefixes
    },
}


def build_faiss_index(chunks, model, model_key, batch_size=64):
    """
    Embed all chunks and build a FAISS index.

    Args:
        chunks: List of chunk dicts (from Phase 2)
        model: Loaded SentenceTransformer model
        model_key: Key in MODEL_CONFIGS ('e5-small' or 'bge-base')
        batch_size: Batch size for embedding

    Returns:
        index: FAISS IndexFlatIP index
        embeddings: numpy array of embeddings
    """
    config = MODEL_CONFIGS[model_key]
    passage_prefix = config['passage_prefix']

    # Prepare texts with appropriate prefix
    texts = [passage_prefix + c['text'] for c in chunks]

    # Embed
    start_time = time.time()
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    embed_time = time.time() - start_time

    # Build FAISS index
    embeddings = np.array(embeddings, dtype='float32')
    embedding_dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(embeddings)

    print(f"  {len(chunks)} chunks → {index.ntotal} vectors ({embedding_dim}d) "
          f"in {embed_time:.1f}s ({len(chunks)/embed_time:.0f} chunks/sec)")

    return index, embeddings


def query_index(query, index, chunks, model, model_key, top_k=5):
    """
    Embed a query and retrieve top-k chunks from a FAISS index.
    Handles model-specific query prefixes automatically.

    Returns: List of dicts with rank, score, metadata, and text preview
    """
    config = MODEL_CONFIGS[model_key]
    prefixed_query = config['query_prefix'] + query

    # Embed query
    query_embedding = model.encode(
        [prefixed_query], normalize_embeddings=True
    ).astype('float32')

    # Search
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        chunk = chunks[idx]
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'meeting_date': chunk['meeting_date'],
            'doc_type': chunk['doc_type'],
            'chunk_index': chunk['chunk_index'],
            'text': chunk['text'],
            'text_preview': chunk['text'][:200],
        })

    return results


print("Embedding and indexing functions defined.")
print(f"Models configured: {list(MODEL_CONFIGS.keys())}")

### Load Embedding Models

In [ ]:
# Load both embedding models
embedding_models = {}

for model_key, config in MODEL_CONFIGS.items():
    print(f"Loading {model_key}: {config['model_name']}...")
    model = SentenceTransformer(config['model_name'])
    embedding_models[model_key] = model
    print(f"  Loaded. Dimension: {model.get_sentence_embedding_dimension()}")

print(f"\nBoth models loaded successfully.")

### Build All Indices

Building 8 FAISS indices: 2 models × 4 chunk sizes.


In [ ]:
# Build FAISS indices for every (model, chunk_size) combination
# Structure: all_indices[model_key][chunk_size] = (index, embeddings)

all_indices = {}
all_embeddings_store = {}

print("Building FAISS indices for all configurations...")
print("=" * 70)

for model_key in MODEL_CONFIGS:
    model = embedding_models[model_key]
    all_indices[model_key] = {}
    all_embeddings_store[model_key] = {}

    print(f"\n{'='*70}")
    print(f"Model: {model_key} ({MODEL_CONFIGS[model_key]['model_name']})")
    print(f"{'='*70}")

    for size in sorted(all_chunk_sets.keys()):
        chunks_for_size = all_chunk_sets[size]
        print(f"\n  Chunk size {size} ({len(chunks_for_size)} chunks):")

        index, embeddings = build_faiss_index(
            chunks_for_size, model, model_key, batch_size=64
        )

        all_indices[model_key][size] = index
        all_embeddings_store[model_key][size] = embeddings

print("\n" + "=" * 70)
print("All indices built!")
print(f"\nSummary:")
for model_key in all_indices:
    sizes = list(all_indices[model_key].keys())
    total = sum(all_indices[model_key][s].ntotal for s in sizes)
    print(f"  {model_key}: {len(sizes)} indices, {total} total vectors")

In [ ]:
# Set primary configuration for the main pipeline
# Default: bge-base with 512-word chunks (strongest expected configuration)
PRIMARY_MODEL = 'bge-base'
PRIMARY_CHUNK_SIZE = 512

index = all_indices[PRIMARY_MODEL][PRIMARY_CHUNK_SIZE]
chunks = all_chunk_sets[PRIMARY_CHUNK_SIZE]
primary_model = embedding_models[PRIMARY_MODEL]

print(f"Primary configuration:")
print(f"  Model: {PRIMARY_MODEL}")
print(f"  Chunk size: {PRIMARY_CHUNK_SIZE} words")
print(f"  Index: {index.ntotal} vectors")

### Sanity Check: Test a Query

In [ ]:
# Test query against primary configuration
test_query = "What was the Fed's stance on inflation in early 2023?"
print(f"Query: \"{test_query}\"")
print(f"Model: {PRIMARY_MODEL} | Chunk size: {PRIMARY_CHUNK_SIZE}")
print("=" * 70)

results = query_index(test_query, index, chunks, primary_model, PRIMARY_MODEL, top_k=5)

for r in results:
    print(f"\n  Rank {r['rank']} (score: {r['score']:.4f})")
    print(f"  Meeting: {r['meeting_date']} | {r['doc_type']} | Chunk {r['chunk_index']}")
    print(f"  Preview: {r['text_preview']}...")

### Compare Retrieval: E5-small vs BGE-base

In [ ]:
# Compare both models on the same query and chunk size
test_query = "What was the Fed's stance on inflation in early 2023?"
compare_chunk_size = 512

print(f"Query: \"{test_query}\"")
print(f"Chunk size: {compare_chunk_size}")
print("=" * 70)

for model_key in MODEL_CONFIGS:
    model = embedding_models[model_key]
    idx = all_indices[model_key][compare_chunk_size]
    chs = all_chunk_sets[compare_chunk_size]

    results = query_index(test_query, idx, chs, model, model_key, top_k=3)

    print(f"\n--- {model_key} ({MODEL_CONFIGS[model_key]['model_name']}) ---")
    for r in results:
        print(f"  Rank {r['rank']} (score: {r['score']:.4f}) | "
              f"Meeting: {r['meeting_date']} | {r['doc_type']}")
        print(f"    {r['text_preview'][:120]}...")

### Compare Retrieval Across Chunk Sizes

In [ ]:
# Compare chunk sizes for BGE-base (our primary model)
test_query = "What reasons did the Fed give for pausing rate hikes?"
model_key = 'bge-base'
model = embedding_models[model_key]

print(f"Query: \"{test_query}\"")
print(f"Model: {model_key}")
print("=" * 70)

for size in sorted(all_indices[model_key].keys()):
    idx = all_indices[model_key][size]
    chs = all_chunk_sets[size]

    results = query_index(test_query, idx, chs, model, model_key, top_k=3)

    print(f"\n--- Chunk size: {size} words ---")
    for r in results:
        print(f"  Rank {r['rank']} (score: {r['score']:.4f}) | "
              f"Meeting: {r['meeting_date']} | Chunk {r['chunk_index']}")
        print(f"    {r['text_preview'][:120]}...")

### Save Indices (Optional)

In [ ]:
# Save all indices to disk in case of runtime restart
import pickle
import os

save_dir = '/content/faiss_indices'
os.makedirs(save_dir, exist_ok=True)

for model_key in all_indices:
    for size in all_indices[model_key]:
        faiss.write_index(
            all_indices[model_key][size],
            f'{save_dir}/index_{model_key}_{size}.faiss'
        )

# Save chunk sets and model configs
with open(f'{save_dir}/chunk_sets.pkl', 'wb') as f:
    pickle.dump(all_chunk_sets, f)

with open(f'{save_dir}/model_configs.pkl', 'wb') as f:
    pickle.dump(MODEL_CONFIGS, f)

total_indices = sum(len(v) for v in all_indices.values())
print(f"Saved {total_indices} indices to {save_dir}/")
print("To reload: faiss.read_index('path') and pickle.load()")

---

## Phase 4: Retrieval

We implement a **hybrid retrieval** system combining three components:

1. **Semantic Search** (FAISS) — finds passages with similar *meaning* to the query
2. **Keyword Search** (BM25) — finds passages containing exact *terms* from the query
3. **Metadata Filtering** — narrows results by date when the query references a time period

**Why hybrid?** Pure semantic search struggles with temporal specificity (as we saw in Phase 3 — a query about "early 2023" retrieved passages from 2021 and 2024). BM25 catches exact date references that semantic search misses. Combining both with metadata filtering gives us the best of both worlds.

**Pipeline flow:**
```
Query → Parse date references → Semantic search (top-20) → Metadata filter
                               → BM25 search (top-20)    → Metadata filter
                                                          → Union & deduplicate → Final top-k
```


In [ ]:
!pip install -q rank_bm25

### Query Date Parser

In [ ]:
import re
from datetime import datetime

def parse_date_from_query(query):
    """
    Extract date range from a natural-language query using regex.

    Handles patterns like:
        - "March 2023" → (2023-03-01, 2023-03-31)
        - "early 2023" → (2023-01-01, 2023-06-30)
        - "late 2024" → (2024-07-01, 2024-12-31)
        - "Q3 2022" → (2022-07-01, 2022-09-30)
        - "2023" (just year) → (2023-01-01, 2023-12-31)
        - "2022 to 2023" or "2022-2023" → (2022-01-01, 2023-12-31)
        - No date found → (None, None)

    Returns: (start_date, end_date) as strings 'YYYY-MM-DD', or (None, None)
    """
    query_lower = query.lower()

    months = {
        'january': 1, 'february': 2, 'march': 3, 'april': 4,
        'may': 5, 'june': 6, 'july': 7, 'august': 8,
        'september': 9, 'october': 10, 'november': 11, 'december': 12,
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
        'jun': 6, 'jul': 7, 'aug': 8, 'sep': 9,
        'oct': 10, 'nov': 11, 'dec': 12,
    }

    import calendar

    # Pattern 1: "Month YYYY" (e.g., "March 2023")
    for month_name, month_num in months.items():
        pattern = rf'{month_name}\s+(\d{{4}})'
        match = re.search(pattern, query_lower)
        if match:
            year = int(match.group(1))
            last_day = calendar.monthrange(year, month_num)[1]
            return f'{year}-{month_num:02d}-01', f'{year}-{month_num:02d}-{last_day:02d}'

    # Pattern 2: "early/late/mid YYYY"
    match = re.search(r'(early|first half of|beginning of)\s+(\d{4})', query_lower)
    if match:
        year = int(match.group(2))
        return f'{year}-01-01', f'{year}-06-30'

    match = re.search(r'(late|second half of|end of)\s+(\d{4})', query_lower)
    if match:
        year = int(match.group(2))
        return f'{year}-07-01', f'{year}-12-31'

    match = re.search(r'(mid|middle of)\s+(\d{4})', query_lower)
    if match:
        year = int(match.group(2))
        return f'{year}-04-01', f'{year}-09-30'

    # Pattern 3: "Q1/Q2/Q3/Q4 YYYY"
    match = re.search(r'q([1-4])\s+(\d{4})', query_lower)
    if match:
        quarter = int(match.group(1))
        year = int(match.group(2))
        start_month = (quarter - 1) * 3 + 1
        end_month = start_month + 2
        last_day = calendar.monthrange(year, end_month)[1]
        return f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-{last_day:02d}'

    # Pattern 4: "YYYY to YYYY" or "YYYY-YYYY"
    match = re.search(r'(\d{4})\s*(?:to|–|-)\s*(\d{4})', query_lower)
    if match:
        year1 = int(match.group(1))
        year2 = int(match.group(2))
        return f'{year1}-01-01', f'{year2}-12-31'

    # Pattern 5: Standalone year "in YYYY" or "during YYYY"
    match = re.search(r'(?:in|during|of)\s+(\d{4})', query_lower)
    if match:
        year = int(match.group(1))
        return f'{year}-01-01', f'{year}-12-31'

    # No date found
    return None, None


# Test the parser
test_queries = [
    "What was the Fed's stance on inflation in March 2023?",
    "How did the FOMC describe the labor market in early 2024?",
    "What happened with rate decisions in Q3 2022?",
    "How did Fed policy evolve from 2022 to 2023?",
    "What reasons did the Fed give for pausing rate hikes?",  # No date
    "What was discussed in late 2024?",
]

print("Date Parser Tests:")
print("-" * 70)
for q in test_queries:
    start, end = parse_date_from_query(q)
    if start:
        print(f"  \"{q[:50]}...\"")
        print(f"    → {start} to {end}")
    else:
        print(f"  \"{q[:50]}...\"")
        print(f"    → No date detected (will search all documents)")

### Metadata Filter

In [ ]:
def filter_by_date(results, start_date, end_date):
    """
    Filter retrieval results by meeting date range.

    Args:
        results: List of result dicts (must contain 'meeting_date' key)
        start_date: 'YYYY-MM-DD' string or None (no filtering)
        end_date: 'YYYY-MM-DD' string or None (no filtering)

    Returns: Filtered list of results
    """
    if start_date is None or end_date is None:
        return results  # No date constraint — return all

    filtered = []
    for r in results:
        meeting_date = r['meeting_date']  # Already 'YYYY-MM-DD' format
        if start_date <= meeting_date <= end_date:
            filtered.append(r)

    return filtered


# Quick test
sample_results = [
    {'meeting_date': '2023-01-31', 'text': 'Jan 2023 meeting'},
    {'meeting_date': '2023-03-22', 'text': 'Mar 2023 meeting'},
    {'meeting_date': '2023-06-14', 'text': 'Jun 2023 meeting'},
    {'meeting_date': '2023-09-20', 'text': 'Sep 2023 meeting'},
    {'meeting_date': '2024-01-31', 'text': 'Jan 2024 meeting'},
]

filtered = filter_by_date(sample_results, '2023-01-01', '2023-06-30')
print("Filter test (early 2023):")
for r in filtered:
    print(f"  {r['meeting_date']}: {r['text']}")

### BM25 Keyword Search

BM25 (Best Matching 25) ranks documents by term frequency, with two key parameters:
- **k1 = 1.5** (default): Controls term frequency saturation — how much credit repeated terms get
- **b = 0.75** (default): Controls length normalization — penalizes longer documents

We use standard defaults, which work well for consistently-structured documents like FOMC minutes. Tuning these on a domain-specific relevance dataset could improve performance and is noted as future work.


In [ ]:
from rank_bm25 import BM25Okapi

def build_bm25_index(chunks):
    """
    Build a BM25 index over chunk texts.

    Tokenization: Simple whitespace + lowercasing.
    For FOMC documents this is sufficient since the text is clean
    and doesn't require stemming or complex tokenization.

    Args:
        chunks: List of chunk dicts

    Returns:
        bm25: BM25Okapi index
    """
    # Tokenize: lowercase and split on whitespace
    tokenized = [c['text'].lower().split() for c in chunks]

    bm25 = BM25Okapi(tokenized)  # Uses default k1=1.5, b=0.75

    print(f"BM25 index built: {len(tokenized)} documents, "
          f"avg length {np.mean([len(t) for t in tokenized]):.0f} tokens")

    return bm25


def bm25_search(query, bm25, chunks, top_k=20):
    """
    Search using BM25 keyword matching.

    Args:
        query: Natural language query string
        bm25: BM25Okapi index
        chunks: List of chunk dicts (same order as used to build index)
        top_k: Number of results to return

    Returns: List of result dicts with rank, score, metadata, text
    """
    # Tokenize query the same way as documents
    query_tokens = query.lower().split()

    # Get BM25 scores for all documents
    scores = bm25.get_scores(query_tokens)

    # Get top-k indices
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices):
        chunk = chunks[idx]
        results.append({
            'rank': rank + 1,
            'score': float(scores[idx]),
            'meeting_date': chunk['meeting_date'],
            'doc_type': chunk['doc_type'],
            'chunk_index': chunk['chunk_index'],
            'text': chunk['text'],
            'text_preview': chunk['text'][:200],
            'source': 'bm25',
        })

    return results


print("BM25 functions defined.")

In [ ]:
# Build BM25 indices for each chunk size
# (BM25 is model-agnostic — only needs one index per chunk size)

all_bm25_indices = {}

print("Building BM25 indices...")
print("-" * 50)

for size in sorted(all_chunk_sets.keys()):
    chs = all_chunk_sets[size]
    print(f"  Chunk size {size}: ", end="")
    all_bm25_indices[size] = build_bm25_index(chs)

print(f"\nAll BM25 indices built: {list(all_bm25_indices.keys())} word sizes")

### Hybrid Retrieval

The core retrieval function that combines semantic search, keyword search, and metadata filtering into a single pipeline.

**Scoring strategy for merging results:** We use Reciprocal Rank Fusion (RRF), which combines rankings from multiple sources without needing to normalize scores across different systems. For each document, its RRF score is:

$$\text{RRF}(d) = \sum_{s \in \text{sources}} \frac{1}{k + \text{rank}_s(d)}$$

where $k$ is a constant (typically 60) that prevents top-ranked documents from dominating.

**Adaptive candidate pool:** When a date filter is active, we expand the initial candidate pool (from 20 to 100) to ensure enough date-relevant chunks survive filtering. At our corpus size, this has negligible compute cost.


In [ ]:
def hybrid_retrieve(query, faiss_index, bm25_index, chunks,
                     embedding_model, model_key,
                     top_k=5, semantic_top_n=20, bm25_top_n=20,
                     rrf_k=60, use_date_filter=True):
    """
    Hybrid retrieval combining semantic search, BM25, and metadata filtering.

    When a date reference is detected in the query:
        - PRE-FILTER: Narrow chunks to the date range FIRST
        - Build a temporary FAISS index from only those chunks
        - Run semantic + BM25 search on the filtered subset

    When no date reference:
        - Search the full corpus with both methods
        - Merge with RRF

    This avoids the problem of relevant-but-temporally-specific chunks
    being buried in a sea of semantically similar chunks from other periods.

    Args:
        query: Natural language query
        faiss_index: FAISS index for semantic search (used when no date filter)
        bm25_index: BM25Okapi index for keyword search (used when no date filter)
        chunks: List of chunk dicts
        embedding_model: Loaded SentenceTransformer
        model_key: Key in MODEL_CONFIGS
        top_k: Final number of results to return
        semantic_top_n: Candidates from semantic search (when no date filter)
        bm25_top_n: Candidates from BM25 search (when no date filter)
        rrf_k: RRF constant (default 60)
        use_date_filter: Whether to apply date-based filtering

    Returns:
        results: List of top-k result dicts, sorted by RRF score
        metadata: Dict with retrieval diagnostics
    """
    # Step 1: Parse date from query
    start_date, end_date = parse_date_from_query(query) if use_date_filter else (None, None)

    # Step 2: Determine search strategy
    if start_date and end_date:
        # --- PRE-FILTER STRATEGY ---
        # Narrow to date-relevant chunks first, then search within that subset

        # Filter chunks by date
        date_filtered_indices = [
            i for i, c in enumerate(chunks)
            if start_date <= c['meeting_date'] <= end_date
        ]
        filtered_chunks = [chunks[i] for i in date_filtered_indices]

        if len(filtered_chunks) == 0:
            # No chunks in date range — fall back to full corpus search
            semantic_results = query_index(
                query, faiss_index, chunks, embedding_model, model_key,
                top_k=semantic_top_n
            )
            for r in semantic_results:
                r['source'] = 'semantic'
            bm25_results = bm25_search(query, bm25_index, chunks, top_k=bm25_top_n)
            semantic_filtered = semantic_results
            bm25_filtered = bm25_results
            filter_note = f"{start_date} to {end_date} (no chunks found, fell back to full corpus)"
        else:
            # Build temporary FAISS index from filtered chunks
            config = MODEL_CONFIGS[model_key]
            passage_prefix = config['passage_prefix']

            filtered_texts = [passage_prefix + c['text'] for c in filtered_chunks]
            filtered_embeddings = embedding_model.encode(
                filtered_texts, normalize_embeddings=True
            ).astype('float32')

            temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
            temp_index.add(filtered_embeddings)

            # Semantic search on filtered subset
            query_prefix = config['query_prefix']
            query_embedding = embedding_model.encode(
                [query_prefix + query], normalize_embeddings=True
            ).astype('float32')

            sem_k = min(semantic_top_n, len(filtered_chunks))
            scores, indices = temp_index.search(query_embedding, sem_k)

            semantic_filtered = []
            for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
                chunk = filtered_chunks[idx]
                semantic_filtered.append({
                    'rank': rank + 1,
                    'score': float(score),
                    'meeting_date': chunk['meeting_date'],
                    'doc_type': chunk['doc_type'],
                    'chunk_index': chunk['chunk_index'],
                    'text': chunk['text'],
                    'text_preview': chunk['text'][:200],
                    'source': 'semantic',
                })

            # BM25 search on filtered subset
            filtered_tokenized = [c['text'].lower().split() for c in filtered_chunks]
            temp_bm25 = BM25Okapi(filtered_tokenized)
            query_tokens = query.lower().split()
            bm25_scores = temp_bm25.get_scores(query_tokens)

            bm25_k = min(bm25_top_n, len(filtered_chunks))
            top_bm25_indices = np.argsort(bm25_scores)[::-1][:bm25_k]

            bm25_filtered = []
            for rank, idx in enumerate(top_bm25_indices):
                chunk = filtered_chunks[idx]
                bm25_filtered.append({
                    'rank': rank + 1,
                    'score': float(bm25_scores[idx]),
                    'meeting_date': chunk['meeting_date'],
                    'doc_type': chunk['doc_type'],
                    'chunk_index': chunk['chunk_index'],
                    'text': chunk['text'],
                    'text_preview': chunk['text'][:200],
                    'source': 'bm25',
                })

            filter_note = f"{start_date} to {end_date} (pre-filtered to {len(filtered_chunks)} chunks)"
    else:
        # --- NO DATE FILTER: search full corpus ---
        semantic_results = query_index(
            query, faiss_index, chunks, embedding_model, model_key,
            top_k=semantic_top_n
        )
        for r in semantic_results:
            r['source'] = 'semantic'

        bm25_results = bm25_search(query, bm25_index, chunks, top_k=bm25_top_n)

        semantic_filtered = semantic_results
        bm25_filtered = bm25_results
        filter_note = "None"

    # Step 3: Reciprocal Rank Fusion
    rrf_scores = {}
    chunk_lookup = {}

    for rank_idx, r in enumerate(semantic_filtered):
        key = (r['meeting_date'], r['chunk_index'], r['doc_type'])
        rrf_score = 1.0 / (rrf_k + rank_idx + 1)
        rrf_scores[key] = rrf_scores.get(key, 0) + rrf_score
        if key not in chunk_lookup:
            chunk_lookup[key] = r

    for rank_idx, r in enumerate(bm25_filtered):
        key = (r['meeting_date'], r['chunk_index'], r['doc_type'])
        rrf_score = 1.0 / (rrf_k + rank_idx + 1)
        rrf_scores[key] = rrf_scores.get(key, 0) + rrf_score
        if key not in chunk_lookup:
            chunk_lookup[key] = r

    # Step 4: Sort by RRF score and return top-k
    sorted_keys = sorted(rrf_scores.keys(), key=lambda k: rrf_scores[k], reverse=True)

    final_results = []
    for rank, key in enumerate(sorted_keys[:top_k]):
        result = chunk_lookup[key].copy()
        result['rrf_score'] = rrf_scores[key]
        result['rank'] = rank + 1
        in_semantic = any(
            (r['meeting_date'], r['chunk_index'], r['doc_type']) == key
            for r in semantic_filtered
        )
        in_bm25 = any(
            (r['meeting_date'], r['chunk_index'], r['doc_type']) == key
            for r in bm25_filtered
        )
        result['found_by'] = []
        if in_semantic:
            result['found_by'].append('semantic')
        if in_bm25:
            result['found_by'].append('bm25')
        final_results.append(result)

    # Diagnostics
    metadata = {
        'query': query,
        'date_filter': filter_note,
        'semantic_candidates': len(semantic_filtered),
        'bm25_candidates': len(bm25_filtered),
        'unique_chunks_merged': len(rrf_scores),
        'final_top_k': len(final_results),
    }

    return final_results, metadata


print("Hybrid retrieval function defined (with pre-filtering).")

### Test Hybrid Retrieval

In [ ]:
def print_retrieval_results(results, metadata):
    """Pretty-print hybrid retrieval results with diagnostics."""
    print(f"Query: \"{metadata['query']}\"")
    print(f"Date filter: {metadata['date_filter']}")
    print(f"Semantic candidates: {metadata['semantic_candidates']}")
    print(f"BM25 candidates: {metadata['bm25_candidates']}")
    print(f"Merged: {metadata['unique_chunks_merged']} unique → "
          f"{metadata['final_top_k']} returned")
    print("-" * 70)

    for r in results:
        sources = ' + '.join(r['found_by'])
        print(f"\n  Rank {r['rank']} (RRF: {r['rrf_score']:.4f}) [{sources}]")
        print(f"  Meeting: {r['meeting_date']} | {r['doc_type']} | Chunk {r['chunk_index']}")
        print(f"  {r['text_preview'][:150]}...")


# Set up primary retrieval components
primary_faiss = all_indices[PRIMARY_MODEL][PRIMARY_CHUNK_SIZE]
primary_bm25 = all_bm25_indices[PRIMARY_CHUNK_SIZE]
primary_chunks = all_chunk_sets[PRIMARY_CHUNK_SIZE]
primary_embed_model = embedding_models[PRIMARY_MODEL]

print("Primary retrieval components set.")

In [ ]:
# Test 1: Query WITH date reference
# Previously returned 0 results — now uses expanded candidate pool
print("=" * 70)
print("TEST 1: Query with date reference (expanded candidate pool)")
print("=" * 70)

results, meta = hybrid_retrieve(
    query="What was the Fed's stance on inflation in early 2023?",
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    top_k=5
)
print_retrieval_results(results, meta)

In [ ]:
# Test 2: Query WITHOUT date reference (pure hybrid, no filtering)
print("=" * 70)
print("TEST 2: Query without date reference")
print("=" * 70)

results, meta = hybrid_retrieve(
    query="What reasons did the Fed give for pausing rate hikes?",
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    top_k=5
)
print_retrieval_results(results, meta)

### Ablation Study: What Does Each Component Contribute?

Compare three retrieval configurations on the same query to isolate the contribution of each component:
1. **Semantic only** — baseline, no BM25, no date filter
2. **Semantic + date filter** — isolates the filter's contribution
3. **Full hybrid** — semantic + BM25 + date filter (shows BM25's additional value)


In [ ]:
query = "What was the Fed's stance on inflation in March 2023?"
print(f"Query: \"{query}\"")
print("=" * 70)

# --- A: Semantic only (baseline) ---
print("\n--- A: SEMANTIC ONLY (no BM25, no date filter) ---")
semantic_only = query_index(
    query, primary_faiss, primary_chunks, primary_embed_model, PRIMARY_MODEL, top_k=5
)
for r in semantic_only:
    print(f"  Rank {r['rank']} (score: {r['score']:.4f}) | "
          f"Meeting: {r['meeting_date']} | {r['doc_type']}")
    print(f"    {r['text_preview'][:120]}...")

# --- B: Semantic + date filter only (no BM25) ---
print("\n\n--- B: SEMANTIC + DATE FILTER (no BM25) ---")
start_date, end_date = parse_date_from_query(query)
print(f"Date filter: {start_date} to {end_date}")

# Get expanded semantic results, then filter
semantic_expanded = query_index(
    query, primary_faiss, primary_chunks, primary_embed_model, PRIMARY_MODEL,
    top_k=100
)
semantic_date_filtered = filter_by_date(semantic_expanded, start_date, end_date)
print(f"100 semantic candidates → {len(semantic_date_filtered)} after date filter")

for r in semantic_date_filtered[:5]:
    print(f"  Rank {r['rank']} (score: {r['score']:.4f}) | "
          f"Meeting: {r['meeting_date']} | {r['doc_type']}")
    print(f"    {r['text_preview'][:120]}...")

# --- C: Full hybrid (semantic + BM25 + date filter) ---
print("\n\n--- C: FULL HYBRID (semantic + BM25 + date filter) ---")
hybrid_results, hybrid_meta = hybrid_retrieve(
    query=query,
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    top_k=5
)
print_retrieval_results(hybrid_results, hybrid_meta)

# Summary
print("\n\n" + "=" * 70)
print("ABLATION SUMMARY")
print("=" * 70)
print(f"  A (Semantic only):        Top meetings = "
      f"{[r['meeting_date'] for r in semantic_only[:3]]}")
print(f"  B (Semantic + filter):    Top meetings = "
      f"{[r['meeting_date'] for r in semantic_date_filtered[:3]]}")
print(f"  C (Full hybrid):          Top meetings = "
      f"{[r['meeting_date'] for r in hybrid_results[:3]]}")
print(f"\n  Target: March 2023 (meeting date: 2023-03-22)")

---

## Phase 5: Generation (Mistral 7B)

Load Mistral 7B with 4-bit quantization and generate grounded answers from retrieved context.

**Why Mistral 7B?** We deliberately chose a lightweight model to demonstrate that RAG enables small open-source models to produce accurate, grounded policy analysis — without API costs or proprietary dependencies. The model's base knowledge of specific FOMC meetings is limited, making retrieval essential rather than supplementary.

**Quantization:** 4-bit quantization via `bitsandbytes` reduces memory from ~14GB (fp16) to ~4GB, fitting comfortably on Colab's free T4 GPU (16GB VRAM).

**Prompt design:** The RAG prompt instructs the model to:
1. Only answer based on the provided context passages
2. Cite which meeting dates it draws from
3. Explicitly state when the context doesn't contain enough information


### Load Mistral 7B

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

def load_mistral():
    """
    Load Mistral 7B Instruct with 4-bit quantization for Colab T4 GPU.

    Returns: (model, tokenizer)
    """
    model_name = "mistralai/Mistral-7B-Instruct-v0.3"

    print(f"Loading {model_name} with 4-bit quantization...")

    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    print(f"Model loaded successfully.")
    print(f"  Device: {model.device}")
    print(f"  Memory: {model.get_memory_footprint() / 1e9:.1f} GB")

    return model, tokenizer

gen_model, tokenizer = load_mistral()

### RAG Prompt Template

In [ ]:
def build_rag_prompt(query, retrieved_results):
    """
    Build the RAG prompt that combines user query with retrieved context.

    The prompt uses Mistral's instruction format and instructs the model to:
    - Only use information from the provided context
    - Cite meeting dates when making claims
    - Acknowledge when context is insufficient

    Args:
        query: User's question
        retrieved_results: List of result dicts from hybrid_retrieve()

    Returns: Formatted prompt string
    """
    # Build context string from retrieved passages
    context_parts = []
    for r in retrieved_results:
        meeting_date = r['meeting_date']
        doc_type = r['doc_type']
        text = r['text']
        context_parts.append(
            f"[Source: FOMC {doc_type}, Meeting date: {meeting_date}]\n{text}"
        )

    context_str = "\n\n---\n\n".join(context_parts)

    # Build the instruction prompt
    system_instruction = (
        "You are a Federal Reserve policy analyst. Answer the user's question "
        "about FOMC monetary policy based ONLY on the provided context passages. "
        "Follow these rules strictly:\n"
        "1. Only state information that is directly supported by the context passages.\n"
        "2. Cite the meeting date when referencing specific information "
        "(e.g., 'In the March 2023 meeting...').\n"
        "3. If the context does not contain enough information to fully answer "
        "the question, explicitly state what is missing.\n"
        "4. Do not add information from your own knowledge — only use the provided context.\n"
        "5. Be concise and specific."
    )

    # Mistral instruction format
    prompt = f"[INST] {system_instruction}\n\n"
    prompt += f"Context passages:\n\n{context_str}\n\n"
    prompt += f"Question: {query} [/INST]"

    return prompt


def build_no_rag_prompt(query):
    """
    Build a prompt WITHOUT retrieval context (for baseline comparison).
    The model must answer from its own knowledge only.
    """
    system_instruction = (
        "You are a Federal Reserve policy analyst. Answer the following question "
        "about FOMC monetary policy. Be specific and cite meeting dates where possible. "
        "Be concise."
    )

    prompt = f"[INST] {system_instruction}\n\nQuestion: {query} [/INST]"

    return prompt


print("Prompt templates defined.")

### Generation Function

In [ ]:
def generate_answer(prompt, model, tokenizer, max_new_tokens=512, temperature=0.3):
    """
    Generate an answer from Mistral 7B given a prompt.

    Args:
        prompt: Formatted prompt string (from build_rag_prompt or build_no_rag_prompt)
        model: Loaded Mistral model
        tokenizer: Loaded tokenizer
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (lower = more deterministic)

    Returns: Generated answer string
    """
    import time

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=4096).to(model.device)

    prompt_length = inputs['input_ids'].shape[1]

    # Generate
    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_time = time.time() - start_time

    # Decode only the new tokens (exclude the prompt)
    answer = tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True
    ).strip()

    tokens_generated = outputs[0].shape[0] - prompt_length

    print(f"  [{tokens_generated} tokens in {gen_time:.1f}s "
          f"({tokens_generated/gen_time:.0f} tok/s) | "
          f"prompt: {prompt_length} tokens]")

    return answer


print("Generation function defined.")

### End-to-End RAG Pipeline

In [ ]:
def rag_pipeline(query, faiss_index, bm25_index, chunks,
                  embedding_model, model_key,
                  gen_model, tokenizer,
                  top_k=5, use_date_filter=True):
    """
    Full end-to-end RAG pipeline: retrieve → build prompt → generate.

    Args:
        query: Natural language question about FOMC policy
        faiss_index: FAISS index for semantic search
        bm25_index: BM25 index for keyword search
        chunks: List of chunk dicts
        embedding_model: Sentence transformer model
        model_key: Key in MODEL_CONFIGS
        gen_model: Loaded Mistral model
        tokenizer: Loaded tokenizer
        top_k: Number of passages to retrieve
        use_date_filter: Whether to apply date filtering

    Returns:
        answer: Generated answer string
        retrieved: List of retrieved passage dicts
        retrieval_meta: Retrieval diagnostics dict
    """
    # Step 1: Retrieve
    retrieved, retrieval_meta = hybrid_retrieve(
        query=query,
        faiss_index=faiss_index,
        bm25_index=bm25_index,
        chunks=chunks,
        embedding_model=embedding_model,
        model_key=model_key,
        top_k=top_k,
        use_date_filter=use_date_filter,
    )

    # Step 2: Build prompt
    if retrieved:
        prompt = build_rag_prompt(query, retrieved)
    else:
        # Fallback: if retrieval returns nothing, use no-RAG prompt
        print("  Warning: No passages retrieved. Falling back to no-RAG prompt.")
        prompt = build_no_rag_prompt(query)

    # Step 3: Generate
    answer = generate_answer(prompt, gen_model, tokenizer)

    return answer, retrieved, retrieval_meta


print("End-to-end RAG pipeline defined.")

### Test the Pipeline

In [ ]:
# Test 1: Date-specific query
print("=" * 70)
print("TEST 1: Date-specific query (RAG)")
print("=" * 70)

query = "What was the Fed's stance on inflation in March 2023?"
print(f"Query: \"{query}\"\n")

answer, retrieved, meta = rag_pipeline(
    query=query,
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    gen_model=gen_model,
    tokenizer=tokenizer,
    top_k=5,
)

print(f"\nRetrieved {len(retrieved)} passages from: "
      f"{[r['meeting_date'] for r in retrieved]}")
print(f"Date filter: {meta['date_filter']}")
print(f"\n{'─'*70}")
print(f"ANSWER:\n{answer}")

In [ ]:
# Test 2: General query (no date)
print("=" * 70)
print("TEST 2: General query (RAG)")
print("=" * 70)

query = "What reasons did the Fed give for pausing rate hikes?"
print(f"Query: \"{query}\"\n")

answer, retrieved, meta = rag_pipeline(
    query=query,
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    gen_model=gen_model,
    tokenizer=tokenizer,
    top_k=5,
)

print(f"\nRetrieved {len(retrieved)} passages from: "
      f"{[r['meeting_date'] for r in retrieved]}")
print(f"\n{'─'*70}")
print(f"ANSWER:\n{answer}")

### With vs Without RAG Comparison

This is the core comparison that demonstrates the pipeline's value: the same model answering the same question, with and without retrieval context.


In [ ]:
# Compare WITH vs WITHOUT RAG on the same query
query = "What was the Fed's stance on inflation in March 2023?"

print("=" * 70)
print(f"Query: \"{query}\"")
print("=" * 70)

# --- WITH RAG ---
print("\n--- WITH RAG ---")
rag_answer, retrieved, meta = rag_pipeline(
    query=query,
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    gen_model=gen_model,
    tokenizer=tokenizer,
    top_k=5,
)
print(f"\nRetrieved from: {[r['meeting_date'] for r in retrieved]}")
print(f"\nRAG ANSWER:\n{rag_answer}")

# --- WITHOUT RAG ---
print("\n\n--- WITHOUT RAG (base model knowledge only) ---")
no_rag_prompt = build_no_rag_prompt(query)
no_rag_answer = generate_answer(no_rag_prompt, gen_model, tokenizer)
print(f"\nBASE ANSWER:\n{no_rag_answer}")

# --- Side by side summary ---
print("\n\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(f"RAG answer length:  {len(rag_answer.split())} words")
print(f"Base answer length: {len(no_rag_answer.split())} words")
print(f"RAG retrieved from: {[r['meeting_date'] for r in retrieved]}")

In [ ]:
# Second comparison: a more specific query
query = "How did the FOMC describe the labor market in early 2024?"

print("=" * 70)
print(f"Query: \"{query}\"")
print("=" * 70)

# --- WITH RAG ---
print("\n--- WITH RAG ---")
rag_answer, retrieved, meta = rag_pipeline(
    query=query,
    faiss_index=primary_faiss,
    bm25_index=primary_bm25,
    chunks=primary_chunks,
    embedding_model=primary_embed_model,
    model_key=PRIMARY_MODEL,
    gen_model=gen_model,
    tokenizer=tokenizer,
    top_k=5,
)
print(f"\nRetrieved from: {[r['meeting_date'] for r in retrieved]}")
print(f"\nRAG ANSWER:\n{rag_answer}")

# --- WITHOUT RAG ---
print("\n\n--- WITHOUT RAG (base model knowledge only) ---")
no_rag_prompt = build_no_rag_prompt(query)
no_rag_answer = generate_answer(no_rag_prompt, gen_model, tokenizer)
print(f"\nBASE ANSWER:\n{no_rag_answer}")

### Debug: Empty Retrieval for "early 2024" Query

In [ ]:
# Step-by-step debug for the failing query
query = "How did the FOMC describe the labor market in early 2024?"

print(f"Query: \"{query}\"")
print("=" * 70)

# Step 1: Check date parsing
start_date, end_date = parse_date_from_query(query)
print(f"\n1. DATE PARSING:")
print(f"   Parsed range: {start_date} to {end_date}")

# Step 2: Check which meetings fall in this range
print(f"\n2. MEETINGS IN DATE RANGE:")
meeting_dates_in_range = sorted(set(
    c['meeting_date'] for c in primary_chunks
    if start_date and start_date <= c['meeting_date'] <= end_date
))
print(f"   Found {len(meeting_dates_in_range)} meetings: {meeting_dates_in_range}")

# Step 3: Count chunks from these meetings
chunks_in_range = [c for c in primary_chunks if c['meeting_date'] in meeting_dates_in_range]
print(f"   Total chunks from these meetings: {len(chunks_in_range)}")

# Step 4: Check semantic search — where do these chunks rank?
print(f"\n3. SEMANTIC SEARCH RANKING:")
all_semantic = query_index(
    query, primary_faiss, primary_chunks, primary_embed_model, PRIMARY_MODEL,
    top_k=len(primary_chunks)  # Search ALL chunks
)

for r in all_semantic:
    if r['meeting_date'] in meeting_dates_in_range:
        print(f"   Rank {r['rank']} (score: {r['score']:.4f}) | "
              f"Meeting: {r['meeting_date']} | {r['doc_type']} | Chunk {r['chunk_index']}")
        if r['rank'] > 200:
            print(f"   ... (stopping, chunks ranked too low)")
            break

# Step 5: Check BM25 — same question
print(f"\n4. BM25 SEARCH RANKING:")
all_bm25 = bm25_search(query, primary_bm25, primary_chunks, top_k=len(primary_chunks))

for r in all_bm25:
    if r['meeting_date'] in meeting_dates_in_range:
        print(f"   Rank {r['rank']} (score: {r['score']:.4f}) | "
              f"Meeting: {r['meeting_date']} | {r['doc_type']} | Chunk {r['chunk_index']}")
        if r['rank'] > 200:
            print(f"   ... (stopping, chunks ranked too low)")
            break

# Step 6: With top-100, how many survive the filter?
print(f"\n5. TOP-100 CANDIDATES → DATE FILTER:")
sem_100 = query_index(query, primary_faiss, primary_chunks, primary_embed_model, PRIMARY_MODEL, top_k=100)
bm25_100 = bm25_search(query, primary_bm25, primary_chunks, top_k=100)

sem_filtered = filter_by_date(sem_100, start_date, end_date)
bm25_filtered = filter_by_date(bm25_100, start_date, end_date)
print(f"   Semantic: 100 → {len(sem_filtered)} after filter")
print(f"   BM25:     100 → {len(bm25_filtered)} after filter")

# Step 7: Try with larger pool
for pool_size in [200, 300, len(primary_chunks)]:
    sem_n = query_index(query, primary_faiss, primary_chunks, primary_embed_model, PRIMARY_MODEL, top_k=pool_size)
    sem_filt = filter_by_date(sem_n, start_date, end_date)
    print(f"   Semantic: {pool_size} → {len(sem_filt)} after filter")

In [ ]:
# Check: what does the actual text of early 2024 labor market chunks look like?
# This helps us understand if the content even discusses "labor market"
print("SAMPLE CHUNKS FROM EARLY 2024 MEETINGS:")
print("=" * 70)

for meeting_date in meeting_dates_in_range[:3]:  # First 3 meetings
    meeting_chunks = [c for c in primary_chunks if c['meeting_date'] == meeting_date]

    # Find chunks that mention labor/employment
    labor_chunks = [c for c in meeting_chunks
                    if any(term in c['text'].lower()
                           for term in ['labor market', 'employment', 'unemployment', 'job gains'])]

    print(f"\n--- {meeting_date} ({len(meeting_chunks)} total chunks, "
          f"{len(labor_chunks)} mention labor/employment) ---")

    for c in labor_chunks[:2]:
        print(f"  Chunk {c['chunk_index']}: {c['text'][:200]}...")

---

## Phase 6: Evaluation

Comprehensive evaluation of the RAG pipeline across multiple dimensions:

1. **Test Query Set** — 15 diverse queries spanning topics, time periods, and difficulty
2. **Retrieval Evaluation** — Precision@k and MRR across 8 configurations (2 models × 4 chunk sizes)
3. **Generation Evaluation** — ROUGE-L, Faithfulness, Answer Relevance, Completeness
4. **With vs Without RAG** — Direct comparison demonstrating the pipeline's value
5. **Claude Haiku as LLM Judge** — Independent evaluation of generation quality


### Setup Claude Haiku Judge

In [ ]:
!pip install -q anthropic

In [ ]:
import anthropic
import time
import json

# Configure Claude as our evaluation judge
# Get your API key from https://console.anthropic.com/
ANTHROPIC_API_KEY = ""  # <-- Replace with your key

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_claude_judge(prompt, max_retries=3):
    """
    Call Claude Haiku as evaluation judge with retry logic.
    Uses claude-haiku-4-5 for cost efficiency.
    Returns the text response.
    """
    for attempt in range(max_retries):
        try:
            message = claude_client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=50,  # We only need short responses (scores)
                messages=[
                    {"role": "user", "content": prompt}
                ],
            )
            return message.content[0].text.strip()
        except anthropic.RateLimitError:
            wait = 2 ** attempt * 5
            print(f"  Rate limited, waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"  Claude error: {e}")
            return None
    return None

# Quick test
test_response = call_claude_judge("Reply with just the word 'working' if you can read this.")
print(f"Claude judge status: {test_response}")

### Test Query Set

15 queries spanning different topics, time periods, and difficulty levels.
Each query has a hand-written reference answer for ROUGE-L scoring.

In [ ]:
test_queries = [
    # --- Date-specific queries (test date filtering) ---
    {
        "query": "What was the Fed's stance on inflation in March 2023?",
        "reference_answer": "In March 2023, the FOMC noted that inflation remains elevated. The Committee raised the target range for the federal funds rate to 4-3/4 to 5 percent and remained highly attentive to inflation risks, anticipating that additional policy firming may be appropriate.",
        "topic": "inflation",
        "has_date": True,
        "difficulty": "easy",
    },
    {
        "query": "How did the FOMC describe the labor market in early 2024?",
        "reference_answer": "In early 2024, the FOMC described job gains as having moderated since early last year but remaining strong, with the unemployment rate remaining low. The Committee judged that risks to achieving employment and inflation goals were moving into better balance.",
        "topic": "labor_market",
        "has_date": True,
        "difficulty": "easy",
    },
    {
        "query": "What was the Fed's interest rate decision in September 2024?",
        "reference_answer": "In September 2024, the FOMC decided to lower the target range for the federal funds rate by 1/2 percentage point to 4-3/4 to 5 percent, reflecting greater confidence that inflation was moving sustainably toward 2 percent and a shift in the balance of risks.",
        "topic": "rate_decision",
        "has_date": True,
        "difficulty": "easy",
    },
    {
        "query": "What did the FOMC say about the housing market in mid 2023?",
        "reference_answer": "In mid 2023, the FOMC noted that the housing sector remained weak, with residential investment continuing to be subdued amid high mortgage rates and elevated home prices that constrained affordability.",
        "topic": "housing",
        "has_date": True,
        "difficulty": "medium",
    },
    {
        "query": "How did the Fed describe economic growth in late 2025?",
        "reference_answer": "In late 2025, the FOMC indicated that economic activity had been expanding at a moderate pace. GDP growth had moderated over the first half of the year, while consumer spending showed some signs of firming.",
        "topic": "economic_growth",
        "has_date": True,
        "difficulty": "medium",
    },

    # --- Topic-specific queries (no date, test semantic retrieval) ---
    {
        "query": "What reasons did the Fed give for raising interest rates?",
        "reference_answer": "The Fed raised interest rates to combat elevated inflation that remained well above its 2 percent target. The Committee judged that ongoing increases in the target range were appropriate to achieve a sufficiently restrictive stance of monetary policy to return inflation to 2 percent over time.",
        "topic": "rate_decision",
        "has_date": False,
        "difficulty": "easy",
    },
    {
        "query": "How did the FOMC assess risks to the economic outlook?",
        "reference_answer": "The FOMC generally assessed that uncertainty about the economic outlook remained elevated, with risks to both sides of its dual mandate. Many participants saw downside risks to employment as having increased, while upside risks to inflation remained elevated.",
        "topic": "risk_assessment",
        "has_date": False,
        "difficulty": "medium",
    },
    {
        "query": "What did the Fed say about the banking sector and financial stability?",
        "reference_answer": "The Fed monitored financial stability risks including elevated asset valuations, hedge fund leverage in Treasury markets, growth of private credit, and potential vulnerabilities from stablecoins. Banks remained resilient with high regulatory capital ratios.",
        "topic": "financial_stability",
        "has_date": False,
        "difficulty": "medium",
    },
    {
        "query": "How did the FOMC discuss the impact of tariffs on the economy?",
        "reference_answer": "The FOMC discussed tariff increases putting upward pressure on inflation through higher goods prices, while also creating uncertainty that weighed on business investment and economic activity. Many participants expected tariff effects on inflation to be temporary but uncertain in magnitude.",
        "topic": "tariffs",
        "has_date": False,
        "difficulty": "medium",
    },
    {
        "query": "What was the Fed's view on wage growth and its relationship to inflation?",
        "reference_answer": "The Fed observed that wage growth had moderated from its peak but remained above levels consistent with 2 percent inflation. Measures like the employment cost index and average hourly earnings showed gradual deceleration, which participants viewed as consistent with a gradually cooling labor market.",
        "topic": "wages",
        "has_date": False,
        "difficulty": "hard",
    },

    # --- Complex/analytical queries ---
    {
        "query": "What factors did the FOMC consider when deciding to begin cutting rates?",
        "reference_answer": "The FOMC considered increased confidence that inflation was moving sustainably toward 2 percent, rising downside risks to employment, a shift in the balance of risks between its dual mandate goals, and the desire to move toward a more neutral policy stance as justification for beginning to cut rates.",
        "topic": "rate_decision",
        "has_date": False,
        "difficulty": "hard",
    },
    {
        "query": "How did the Fed's balance sheet policy evolve over this period?",
        "reference_answer": "The Fed reduced its securities holdings beginning in 2022, slowed the pace of reduction in 2024, and concluded balance sheet runoff in late 2025. It then initiated reserve management purchases of shorter-term Treasury securities to maintain ample reserves.",
        "topic": "balance_sheet",
        "has_date": False,
        "difficulty": "hard",
    },
    {
        "query": "What concerns did FOMC participants express about inflation expectations becoming unanchored?",
        "reference_answer": "Participants emphasized the importance of maintaining well-anchored longer-term inflation expectations. Many expressed concern that a prolonged period of above-target inflation could risk an increase in longer-run expectations, and that premature easing could be misinterpreted as diminished commitment to the 2 percent objective.",
        "topic": "inflation_expectations",
        "has_date": False,
        "difficulty": "hard",
    },
    {
        "query": "How did the Fed respond to the COVID-19 pandemic's economic impact in 2020?",
        "reference_answer": "The Fed responded to COVID-19 by cutting rates to near zero, launching large-scale asset purchases, and establishing emergency lending facilities. The FOMC committed to maintaining accommodative policy until the economy was on track to achieve maximum employment and price stability goals.",
        "topic": "covid_response",
        "has_date": True,
        "difficulty": "medium",
    },
    {
        "query": "What role did artificial intelligence play in FOMC economic discussions?",
        "reference_answer": "The FOMC discussed AI primarily in the context of its potential effects on productivity growth, business investment, and labor demand. Participants noted strong investment in AI-related technology by large companies, and several suggested that AI-driven productivity gains could boost economic growth without generating inflationary pressures.",
        "topic": "AI",
        "has_date": False,
        "difficulty": "hard",
    },
]

print(f"Test query set: {len(test_queries)} queries")
print(f"  By difficulty: {pd.Series([q['difficulty'] for q in test_queries]).value_counts().to_dict()}")
print(f"  By topic: {pd.Series([q['topic'] for q in test_queries]).value_counts().to_dict()}")
print(f"  With date: {sum(1 for q in test_queries if q['has_date'])}, Without: {sum(1 for q in test_queries if not q['has_date'])}")

### Run RAG Pipeline on All Test Queries

Generate answers with and without RAG for all test queries using the primary configuration (BGE-base, 512-word chunks).

In [ ]:
# Run all test queries through the pipeline — WITH and WITHOUT RAG
all_results = []

print("Running RAG pipeline on all test queries...")
print("=" * 70)

for i, tq in enumerate(test_queries):
    query = tq['query']
    print(f"\n[{i+1}/{len(test_queries)}] {query[:60]}...")

    # WITH RAG
    print("  Generating WITH RAG...")
    rag_answer, retrieved, retrieval_meta = rag_pipeline(
        query=query,
        faiss_index=primary_faiss,
        bm25_index=primary_bm25,
        chunks=primary_chunks,
        embedding_model=primary_embed_model,
        model_key=PRIMARY_MODEL,
        gen_model=gen_model,
        tokenizer=tokenizer,
        top_k=5,
    )

    # WITHOUT RAG
    print("  Generating WITHOUT RAG...")
    no_rag_prompt = build_no_rag_prompt(query)
    no_rag_answer = generate_answer(no_rag_prompt, gen_model, tokenizer)

    all_results.append({
        'query': query,
        'reference_answer': tq['reference_answer'],
        'topic': tq['topic'],
        'has_date': tq['has_date'],
        'difficulty': tq['difficulty'],
        'rag_answer': rag_answer,
        'no_rag_answer': no_rag_answer,
        'retrieved_passages': retrieved,
        'retrieval_meta': retrieval_meta,
        'retrieved_dates': [r['meeting_date'] for r in retrieved],
    })

print("\n" + "=" * 70)
print(f"Complete! Generated {len(all_results)} RAG + {len(all_results)} base answers.")

### ROUGE-L Evaluation

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

for r in all_results:
    # ROUGE-L: RAG answer vs reference
    rag_rouge = scorer.score(r['reference_answer'], r['rag_answer'])
    r['rag_rouge_l'] = rag_rouge['rougeL'].fmeasure

    # ROUGE-L: Base answer vs reference
    base_rouge = scorer.score(r['reference_answer'], r['no_rag_answer'])
    r['base_rouge_l'] = base_rouge['rougeL'].fmeasure

# Summary
rag_scores = [r['rag_rouge_l'] for r in all_results]
base_scores = [r['base_rouge_l'] for r in all_results]

print("ROUGE-L Scores (vs reference answers)")
print("=" * 50)
print(f"  RAG:  mean={np.mean(rag_scores):.3f}, median={np.median(rag_scores):.3f}")
print(f"  Base: mean={np.mean(base_scores):.3f}, median={np.median(base_scores):.3f}")
print(f"\n  Per query:")
for r in all_results:
    print(f"    {r['query'][:50]:50s} RAG={r['rag_rouge_l']:.3f}  Base={r['base_rouge_l']:.3f}")

### LLM Judge Evaluation (Claude Haiku)

Score each answer on Faithfulness (1-5), Answer Relevance (1-5), and Completeness (1-5).

In [ ]:
def judge_answer(query, answer, retrieved_chunks, metric):
    """
    Use Claude Haiku to score a generated answer.
    Returns integer score 1-5.
    """
    context_str = ""
    if retrieved_chunks:
        context_parts = []
        for r in retrieved_chunks:
            context_parts.append(f"[{r['meeting_date']}, {r['doc_type']}]: {r['text'][:500]}")
        context_str = "\n\n".join(context_parts)

    prompts = {
        "faithfulness": f"""Rate the faithfulness of this answer on a scale of 1-5.
A faithful answer ONLY contains information supported by the provided context.

Scoring:
5 = Every claim is directly supported by the context
4 = Almost all claims supported, minor unsupported details
3 = Most claims supported but some notable unsupported additions
2 = Significant unsupported claims mixed with supported ones
1 = Mostly unsupported or hallucinated information

Context passages:
{context_str}

Question: {query}
Answer: {answer}

Reply with ONLY a single number (1-5):""",

        "relevance": f"""Rate how relevant this answer is to the question on a scale of 1-5.

Scoring:
5 = Directly and completely addresses the question
4 = Mostly addresses the question with minor gaps
3 = Partially addresses the question
2 = Tangentially related but doesn't answer the question
1 = Completely off-topic

Question: {query}
Answer: {answer}

Reply with ONLY a single number (1-5):""",

        "completeness": f"""Rate the completeness of this answer on a scale of 1-5.
A complete answer covers the key points relevant to the question.

Scoring:
5 = Covers all major points comprehensively
4 = Covers most key points with minor omissions
3 = Covers some key points but misses important information
2 = Only covers a small portion of relevant information
1 = Misses nearly all key information

Question: {query}
Answer: {answer}

Reply with ONLY a single number (1-5):"""
    }

    response = call_claude_judge(prompts[metric])

    if response:
        # Extract the number from response
        import re
        numbers = re.findall(r'[1-5]', response)
        if numbers:
            return int(numbers[0])

    return None


print("Judge function defined.")

In [ ]:
# Run LLM judge on all answers
metrics = ['faithfulness', 'relevance', 'completeness']

print("Running Claude Haiku evaluation...")
print("=" * 70)

for i, r in enumerate(all_results):
    print(f"\n[{i+1}/{len(all_results)}] {r['query'][:50]}...")

    for metric in metrics:
        # Score RAG answer
        rag_score = judge_answer(
            r['query'], r['rag_answer'], r['retrieved_passages'], metric
        )
        r[f'rag_{metric}'] = rag_score

        # Score base answer (no context for faithfulness — judge on general accuracy)
        base_score = judge_answer(
            r['query'], r['no_rag_answer'], [], metric
        )
        r[f'base_{metric}'] = base_score

        print(f"  {metric}: RAG={rag_score}, Base={base_score}")

        # Rate limiting pause
        time.sleep(1)

print("\n" + "=" * 70)
print("LLM judge evaluation complete!")

### Retrieval Evaluation: Precision@k and MRR

Use Claude Haiku to judge relevance of each retrieved passage, then compute Precision@k and MRR.

In [ ]:
def judge_retrieval_relevance(query, chunk_text, meeting_date, doc_type):
    """
    Use Claude Haiku to judge if a retrieved chunk is relevant to the query.
    Returns: 'relevant', 'partially_relevant', or 'not_relevant'
    """
    prompt = f"""Given the following query and retrieved passage, rate the relevance.

- RELEVANT: The passage directly answers or informs the query
- PARTIALLY RELEVANT: The passage is related but doesn't directly answer
- NOT RELEVANT: The passage is unrelated to the query

Query: {query}
Passage (from FOMC {doc_type}, {meeting_date}): {chunk_text[:800]}

Reply with ONLY one of: RELEVANT, PARTIALLY RELEVANT, NOT RELEVANT"""

    response = call_claude_judge(prompt)

    if response:
        response_lower = response.lower().strip()
        if 'not relevant' in response_lower or 'not_relevant' in response_lower:
            return 'not_relevant'
        elif 'partially' in response_lower:
            return 'partially_relevant'
        elif 'relevant' in response_lower:
            return 'relevant'

    return None


# Evaluate retrieval for all queries
print("Evaluating retrieval quality...")
print("=" * 70)

for i, r in enumerate(all_results):
    print(f"\n[{i+1}/{len(all_results)}] {r['query'][:50]}...")

    relevance_judgments = []

    for passage in r['retrieved_passages']:
        judgment = judge_retrieval_relevance(
            r['query'], passage['text'], passage['meeting_date'], passage['doc_type']
        )
        relevance_judgments.append(judgment)
        time.sleep(0.5)  # Rate limiting

    r['relevance_judgments'] = relevance_judgments

    # Compute Precision@k
    if relevance_judgments:
        relevant_count = sum(
            1 for j in relevance_judgments if j == 'relevant'
        ) + 0.5 * sum(
            1 for j in relevance_judgments if j == 'partially_relevant'
        )
        r['precision_at_k'] = relevant_count / len(relevance_judgments)
    else:
        r['precision_at_k'] = 0.0

    # Compute MRR
    r['mrr'] = 0.0
    for rank, j in enumerate(relevance_judgments):
        if j in ('relevant', 'partially_relevant'):
            r['mrr'] = 1.0 / (rank + 1)
            break

    print(f"  Judgments: {relevance_judgments}")
    print(f"  Precision@{len(relevance_judgments)}: {r['precision_at_k']:.2f}, MRR: {r['mrr']:.2f}")

print("\n" + "=" * 70)
print("Retrieval evaluation complete!")

### Results Summary

Aggregate all evaluation metrics into tables and visualizations.

In [ ]:
# Build results DataFrame
results_df = pd.DataFrame([{
    'query': r['query'][:50],
    'topic': r['topic'],
    'has_date': r['has_date'],
    'difficulty': r['difficulty'],
    'rag_rouge_l': r.get('rag_rouge_l', None),
    'base_rouge_l': r.get('base_rouge_l', None),
    'rag_faithfulness': r.get('rag_faithfulness', None),
    'base_faithfulness': r.get('base_faithfulness', None),
    'rag_relevance': r.get('rag_relevance', None),
    'base_relevance': r.get('base_relevance', None),
    'rag_completeness': r.get('rag_completeness', None),
    'base_completeness': r.get('base_completeness', None),
    'precision_at_k': r.get('precision_at_k', None),
    'mrr': r.get('mrr', None),
    'num_retrieved': len(r['retrieved_passages']),
} for r in all_results])

print("OVERALL RESULTS SUMMARY")
print("=" * 70)

# RAG vs Base comparison
print("\n--- RAG vs Base (mean scores) ---")
comparison_metrics = [
    ('ROUGE-L', 'rag_rouge_l', 'base_rouge_l'),
    ('Faithfulness', 'rag_faithfulness', 'base_faithfulness'),
    ('Relevance', 'rag_relevance', 'base_relevance'),
    ('Completeness', 'rag_completeness', 'base_completeness'),
]

print(f"\n{'Metric':<20} {'RAG':>10} {'Base':>10} {'Δ':>10}")
print("-" * 50)
for name, rag_col, base_col in comparison_metrics:
    rag_mean = results_df[rag_col].mean()
    base_mean = results_df[base_col].mean()
    delta = rag_mean - base_mean
    print(f"{name:<20} {rag_mean:>10.3f} {base_mean:>10.3f} {delta:>+10.3f}")

# Retrieval metrics
print(f"\n--- Retrieval Metrics ---")
print(f"Mean Precision@k: {results_df['precision_at_k'].mean():.3f}")
print(f"Mean MRR:         {results_df['mrr'].mean():.3f}")

# By difficulty
print(f"\n--- RAG ROUGE-L by Difficulty ---")
for diff in ['easy', 'medium', 'hard']:
    subset = results_df[results_df['difficulty'] == diff]
    print(f"  {diff}: {subset['rag_rouge_l'].mean():.3f} (n={len(subset)})")

# By date presence
print(f"\n--- Retrieval by Date Presence ---")
for has_date in [True, False]:
    subset = results_df[results_df['has_date'] == has_date]
    label = "With date" if has_date else "Without date"
    print(f"  {label}: Precision@k={subset['precision_at_k'].mean():.3f}, "
          f"MRR={subset['mrr'].mean():.3f} (n={len(subset)})")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. RAG vs Base: bar chart of mean scores
ax = axes[0, 0]
metrics_names = ['ROUGE-L', 'Faithfulness', 'Relevance', 'Completeness']
rag_means = [results_df[f'rag_{m}'].mean() for m in ['rouge_l', 'faithfulness', 'relevance', 'completeness']]
base_means = [results_df[f'base_{m}'].mean() for m in ['rouge_l', 'faithfulness', 'relevance', 'completeness']]

x = np.arange(len(metrics_names))
width = 0.35
ax.bar(x - width/2, rag_means, width, label='RAG', color='steelblue')
ax.bar(x + width/2, base_means, width, label='Base', color='coral')
ax.set_ylabel('Score')
ax.set_title('RAG vs Base: Mean Scores')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 5.5)

# 2. Retrieval metrics by topic
ax = axes[0, 1]
topic_precision = results_df.groupby('topic')['precision_at_k'].mean().sort_values()
topic_precision.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Mean Precision@k')
ax.set_title('Retrieval Precision by Topic')

# 3. ROUGE-L distribution
ax = axes[1, 0]
ax.hist(results_df['rag_rouge_l'].dropna(), bins=10, alpha=0.7, label='RAG', color='steelblue')
ax.hist(results_df['base_rouge_l'].dropna(), bins=10, alpha=0.7, label='Base', color='coral')
ax.set_xlabel('ROUGE-L Score')
ax.set_ylabel('Frequency')
ax.set_title('ROUGE-L Score Distribution')
ax.legend()

# 4. Per-query comparison
ax = axes[1, 1]
query_labels = [f"Q{i+1}" for i in range(len(all_results))]
ax.plot(query_labels, results_df['rag_rouge_l'], 'o-', label='RAG', color='steelblue')
ax.plot(query_labels, results_df['base_rouge_l'], 's-', label='Base', color='coral')
ax.set_xlabel('Query')
ax.set_ylabel('ROUGE-L')
ax.set_title('Per-Query ROUGE-L Comparison')
ax.legend()
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Chunk Size Comparison

Evaluate retrieval quality across 4 chunk sizes using the primary embedding model (BGE-base).

In [ ]:
# Run retrieval across all chunk sizes with FULL top-5 evaluation
chunk_size_queries = test_queries[:8]  # First 8 queries

chunk_size_results = {}

for size in sorted(all_chunk_sets.keys()):
    print(f"\n--- Chunk size: {size} ---")
    faiss_idx = all_indices[PRIMARY_MODEL][size]
    bm25_idx = all_bm25_indices[size]
    chs = all_chunk_sets[size]

    size_results = []

    for tq in chunk_size_queries:
        results, meta = hybrid_retrieve(
            query=tq['query'],
            faiss_index=faiss_idx,
            bm25_index=bm25_idx,
            chunks=chs,
            embedding_model=primary_embed_model,
            model_key=PRIMARY_MODEL,
            top_k=5,
        )

        # Judge ALL retrieved passages (not just top-1)
        relevance_judgments = []
        for passage in results:
            judgment = judge_retrieval_relevance(
                tq['query'], passage['text'],
                passage['meeting_date'], passage['doc_type']
            )
            relevance_judgments.append(judgment)
            time.sleep(0.5)  # Rate limiting

        # Compute Precision@k
        if relevance_judgments:
            relevant_count = sum(
                1 for j in relevance_judgments if j == 'relevant'
            ) + 0.5 * sum(
                1 for j in relevance_judgments if j == 'partially_relevant'
            )
            precision_at_k = relevant_count / len(relevance_judgments)
        else:
            precision_at_k = 0.0

        # Compute MRR
        mrr = 0.0
        for rank, j in enumerate(relevance_judgments):
            if j in ('relevant', 'partially_relevant'):
                mrr = 1.0 / (rank + 1)
                break

        size_results.append({
            'query': tq['query'][:50],
            'num_retrieved': len(results),
            'judgments': relevance_judgments,
            'precision_at_k': precision_at_k,
            'mrr': mrr,
        })

        print(f"  {tq['query'][:50]}...")
        print(f"    Judgments: {relevance_judgments}")
        print(f"    P@{len(relevance_judgments)}={precision_at_k:.2f}, MRR={mrr:.2f}")

    chunk_size_results[size] = size_results

# Summary table
print("\n\nCHUNK SIZE COMPARISON (Top-5 Evaluation)")
print("=" * 70)
print(f"\n{'Size':<10} {'Avg Retrieved':<15} {'Mean P@k':<15} {'Mean MRR':<15}")
print("-" * 55)

for size in sorted(chunk_size_results.keys()):
    results = chunk_size_results[size]
    avg_retrieved = np.mean([r['num_retrieved'] for r in results])
    mean_precision = np.mean([r['precision_at_k'] for r in results])
    mean_mrr = np.mean([r['mrr'] for r in results])
    print(f"{size:<10} {avg_retrieved:<15.1f} {mean_precision:<15.3f} {mean_mrr:<15.3f}")

### Embedding Model Comparison

Compare E5-small vs BGE-base retrieval quality.

In [ ]:
# Compare embedding models with FULL top-5 evaluation on 512 chunk size
model_comparison_queries = test_queries[:8]

model_results = {}

for model_key in MODEL_CONFIGS:
    print(f"\n--- Model: {model_key} ---")
    faiss_idx = all_indices[model_key][512]
    bm25_idx = all_bm25_indices[512]
    chs = all_chunk_sets[512]
    model = embedding_models[model_key]

    m_results = []

    for tq in model_comparison_queries:
        results, meta = hybrid_retrieve(
            query=tq['query'],
            faiss_index=faiss_idx,
            bm25_index=bm25_idx,
            chunks=chs,
            embedding_model=model,
            model_key=model_key,
            top_k=5,
        )

        # Judge ALL retrieved passages
        relevance_judgments = []
        for passage in results:
            judgment = judge_retrieval_relevance(
                tq['query'], passage['text'],
                passage['meeting_date'], passage['doc_type']
            )
            relevance_judgments.append(judgment)
            time.sleep(0.5)

        # Precision@k
        if relevance_judgments:
            relevant_count = sum(
                1 for j in relevance_judgments if j == 'relevant'
            ) + 0.5 * sum(
                1 for j in relevance_judgments if j == 'partially_relevant'
            )
            precision_at_k = relevant_count / len(relevance_judgments)
        else:
            precision_at_k = 0.0

        # MRR
        mrr = 0.0
        for rank, j in enumerate(relevance_judgments):
            if j in ('relevant', 'partially_relevant'):
                mrr = 1.0 / (rank + 1)
                break

        m_results.append({
            'query': tq['query'][:50],
            'num_retrieved': len(results),
            'judgments': relevance_judgments,
            'precision_at_k': precision_at_k,
            'mrr': mrr,
        })

        print(f"  {tq['query'][:50]}...")
        print(f"    Judgments: {relevance_judgments}")
        print(f"    P@{len(relevance_judgments)}={precision_at_k:.2f}, MRR={mrr:.2f}")

    model_results[model_key] = m_results

# Summary
print("\n\nEMBEDDING MODEL COMPARISON (Top-5 Evaluation)")
print("=" * 70)
print(f"\n{'Model':<15} {'Avg Retrieved':<15} {'Mean P@k':<15} {'Mean MRR':<15}")
print("-" * 60)

for model_key in model_results:
    results = model_results[model_key]
    avg_retrieved = np.mean([r['num_retrieved'] for r in results])
    mean_precision = np.mean([r['precision_at_k'] for r in results])
    mean_mrr = np.mean([r['mrr'] for r in results])
    print(f"{model_key:<15} {avg_retrieved:<15.1f} {mean_precision:<15.3f} {mean_mrr:<15.3f}")

### Save All Results

In [ ]:
# Save results for report writing
import json

# Convert to serializable format
save_results = []
for r in all_results:
    save_r = {k: v for k, v in r.items() if k != 'retrieved_passages'}
    save_r['num_passages'] = len(r['retrieved_passages'])
    save_results.append(save_r)

with open('/content/drive/MyDrive/Colab Notebooks/evaluation_results.csv', 'w') as f:
    json.dump(save_results, f, indent=2, default=str)

results_df.to_csv('/content/drive/MyDrive/Colab Notebooks/evaluation_results.csv', index=False)

print(f"Results saved:")
print(f"  /content/evaluation_results.json")
print(f"  /content/evaluation_results.csv")

# Final summary
print(f"\n{'='*70}")
print(f"EVALUATION COMPLETE")
print(f"{'='*70}")
print(f"Queries evaluated: {len(all_results)}")
print(f"Metrics computed: ROUGE-L, Faithfulness, Relevance, Completeness, Precision@k, MRR")
print(f"Configurations compared: RAG vs Base, 4 chunk sizes, 2 embedding models")

---
## Phase 7 (Optional): RoBERTa Stance Enrichment
*If time permits — integrate fine-tuned RoBERTa for hawkish/dovish classification.*


In [ ]:
# Phase 7 code will go here